In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/2025-sep-dl-gen-ai-project/sample_submission.csv
/kaggle/input/2025-sep-dl-gen-ai-project/train.csv
/kaggle/input/2025-sep-dl-gen-ai-project/test.csv


Loaded train.csv,test.csv

Vectorized text with TF-IDF (1–2 gram, 40k features, sublinear TF).

Trained a One-vs-Rest LinearSVC (works great for sparse text; no probs with class imbalance).

Validated on a 15% hold-out split → Macro F1 ≈ 0.748.

Refit on full training data and generated predictions for the test set.

Wrote a submission file in the exact format (id + five emotion columns).

In [3]:
import transformers, torch, sys
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)


transformers: 4.52.4
torch: 2.6.0+cu124


In [4]:
# K0 — Setup + W&B login
import os, random, numpy as np, torch, transformers, inspect
from pathlib import Path
import pandas as pd

# Quiet logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Repro
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| transformers:", transformers.__version__)

# Data paths
DATA_DIR = Path("/kaggle/input/2025-sep-dl-gen-ai-project")
TRAIN = DATA_DIR / "train.csv"
TEST  = DATA_DIR / "test.csv"
SAMPLE_SUB = DATA_DIR / "sample_submission.csv"
for p in [TRAIN, TEST, SAMPLE_SUB]:
    assert p.exists(), f"Missing: {p}"

# Labels/cols
ID_COL, TEXT_COL = "id", "text"
LABELS = ["anger","fear","joy","sadness","surprise"]
id2label = {i:l for i,l in enumerate(LABELS)}
label2id = {l:i for i,l in enumerate(LABELS)}

# Hyperparams
MODEL_NAME   = "roberta-base"
MAX_LEN      = 160
BATCH_SIZE   = 16  # lower to 12/8 if OOM
EPOCHS       = 4
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

# ---- W&B login via Kaggle Secret ----
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("WANDB_API_KEY")
    assert key and len(key) > 25, "WANDB_API_KEY missing/invalid in Kaggle Secrets"
    os.environ["WANDB_API_KEY"] = key
    os.environ["WANDB_START_METHOD"] = "thread"
    os.environ["WANDB_INIT_TIMEOUT"] = "60"
    wandb.login(key=key, relogin=True)
    run = wandb.init(
        project="kaggle-emotions",
        name=f"roberta_f1_tuned_{SEED}",
        config=dict(
            model=MODEL_NAME, max_len=MAX_LEN, epochs=EPOCHS,
            batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO
        ),
        settings=wandb.Settings(start_method="thread", init_timeout=60),
        reinit=True
    )
    print("W&B URL:", run.url)
except Exception as e:
    raise RuntimeError(f"W&B init failed: {e}")


Device: cuda | transformers: 4.52.4


wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 21f3000279 (21f3000279-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


W&B URL: https://wandb.ai/21f3000279-iit-madras/kaggle-emotions/runs/o69xwwor


In [5]:
# --- FIXED: rebuild datasets if needed, then build Trainer correctly and train ---

import inspect, re, torch, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.nn import BCEWithLogitsLoss
from torch.utils.data import Dataset

# ===== Ensure tokenizer =====
if "tokenizer" not in globals() or isinstance(tokenizer, str):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# ===== Ensure train_df/test_df =====
if "train_df" not in globals() or "test_df" not in globals():
    DATA_DIR = Path("/kaggle/input/2025-sep-dl-gen-ai-project")
    TRAIN = DATA_DIR / "train.csv"
    TEST  = DATA_DIR / "test.csv"
    train_df = pd.read_csv(TRAIN)
    test_df  = pd.read_csv(TEST)
    def clean_text(s):
        if not isinstance(s, str): s = "" if pd.isna(s) else str(s)
        s = s.lower(); s = re.sub(r"\s+", " ", s).strip()
        return s
    train_df["text"] = train_df["text"].apply(clean_text)
    test_df["text"]  = test_df["text"].apply(clean_text)

# ===== Torch datasets (if missing) =====
class EmotionTrainDS(Dataset):
    def __init__(self, df, text_col, labels, tok, max_len):
        self.df=df.reset_index(drop=True); self.text_col=text_col; self.labels=labels
        self.tok=tok; self.max_len=max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        text = " ".join(str(self.df.loc[idx, self.text_col]).split())
        enc  = self.tok(text, truncation=True, padding="max_length",
                        max_length=self.max_len, return_tensors="pt")
        item = {k: v.squeeze(0) for k,v in enc.items()}
        y    = torch.tensor(self.df.loc[idx, self.labels].values.astype("float32"))
        item["labels"] = y
        return item

if "ds_train" not in globals() or "ds_val" not in globals():
    LABELS = ["anger","fear","joy","sadness","surprise"]
    label_counts = train_df[LABELS].sum(axis=1).clip(upper=len(LABELS))
    tr_df, va_df = train_test_split(train_df, test_size=0.2, random_state=SEED, stratify=label_counts)
    ds_train = EmotionTrainDS(tr_df, "text", LABELS, tokenizer, MAX_LEN)
    ds_val   = EmotionTrainDS(va_df,  "text", LABELS, tokenizer, MAX_LEN)

# ===== Ensure model object (not a string) =====
if ("model" not in globals()) or isinstance(model, str):
    id2label = {i:l for i,l in enumerate(LABELS)}
    label2id = {l:i for i,l in enumerate(LABELS)}
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        problem_type="multi_label_classification",
        id2label=id2label, label2id=label2id
    )
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

# ===== Ensure pos_weight tensor =====
if "pos_weight" not in globals():
    y_all = train_df[LABELS].astype(int).values
    pos = y_all.sum(axis=0); neg = y_all.shape[0] - pos
    pos_weight = torch.tensor((neg / (pos.clip(min=1)))).float()

# ===== Version-safe TrainingArguments =====
ta_kwargs = dict(
    output_dir="/kaggle/working/roberta_mlc",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=EPOCHS,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    warmup_ratio=WARMUP_RATIO,
    report_to=["wandb"],
)
if "evaluation_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    ta_kwargs["evaluation_strategy"] = "epoch"
else:
    ta_kwargs["eval_strategy"] = "epoch"

training_args = TrainingArguments(**ta_kwargs)

# ===== v5-safe Trainer =====
class PosWeightTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        self.pos_weight = pos_weight
        super().__init__(*args, **kwargs)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels_t = inputs.pop("labels")
        outputs  = model(**inputs)
        logits   = outputs.logits
        if not torch.is_floating_point(labels_t):
            labels_t = labels_t.float()
        loss_fn = BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))
        loss = loss_fn(logits, labels_t.to(logits.device))
        return (loss, outputs) if return_outputs else loss

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=ds_train,   # ✅ correct
    eval_dataset=ds_val,      # ✅ correct
    compute_metrics=lambda _: {},
    pos_weight=pos_weight,    # ✅ tensor, NOT weight decay
)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = PosWeightTrainer(**trainer_kwargs)

print(">> Training …")
trainer.train()
print(">> Training complete.")


E0000 00:00:1760096098.836103      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760096098.894208      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


>> Training …


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss
1,0.920800,0.588356
2,0.534200,0.499255
3,0.386000,0.466487
4,0.341900,0.454222


>> Training complete.


In [6]:
# K2 (robust): rebuild ds_test if missing → tune thresholds → predict → save
import numpy as np, pandas as pd, torch, json, re
from pathlib import Path
from sklearn.metrics import f1_score, classification_report

# ---- Required from K1: trainer + ds_val + va_df ----
assert "trainer" in globals(), "Missing trainer. Run K1 first."
assert "ds_val" in globals(), "Missing ds_val. Run K1 first."
assert "va_df" in globals(), "Missing va_df. Run K1 first."

# ---- Ensure shared constants ----
LABELS   = globals().get("LABELS", ["anger","fear","joy","sadness","surprise"])
ID_COL   = globals().get("ID_COL", "id")
TEXT_COL = globals().get("TEXT_COL", "text")
MAX_LEN  = globals().get("MAX_LEN", 160)

# ---- Ensure tokenizer ----
from transformers import AutoTokenizer
if "tokenizer" not in globals():
    MODEL_NAME = globals().get("MODEL_NAME", "roberta-base")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# ---- Ensure test_df present; else load + clean from competition input ----
if "test_df" not in globals():
    COMP_DIR = Path("/kaggle/input/2025-sep-dl-gen-ai-project")
    TEST_csv = COMP_DIR / "test.csv"
    assert TEST_csv.exists(), "test.csv not found. Add competition dataset as input."
    test_df = pd.read_csv(TEST_csv)

# Ensure clean_text exists and text is cleaned (match training)
if "clean_text" not in globals():
    def clean_text(s):
        if not isinstance(s, str):
            s = "" if pd.isna(s) else str(s)
        s = s.lower()
        s = re.sub(r"\s+", " ", s).strip()
        return s
test_df[TEXT_COL] = test_df[TEXT_COL].apply(clean_text)

# ---- Rebuild ds_test if missing ----
from torch.utils.data import Dataset

if "EmotionTestDS" not in globals():
    class EmotionTestDS(Dataset):
        def __init__(self, df, text_col, tok, max_len):
            self.df=df.reset_index(drop=True); self.text_col=text_col
            self.tok=tok; self.max_len=max_len
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            text = " ".join(str(self.df.loc[idx, self.text_col]).split())
            enc  = self.tok(text, truncation=True, padding="max_length",
                            max_length=self.max_len, return_tensors="pt")
            return {k: v.squeeze(0) for k,v in enc.items()}

if "ds_test" not in globals():
    ds_test = EmotionTestDS(test_df, TEXT_COL, tokenizer, MAX_LEN)

# ================== Threshold tuning on validation ==================
val_pred   = trainer.predict(ds_val)
val_logits = val_pred.predictions
val_probs  = torch.sigmoid(torch.tensor(val_logits)).numpy()
y_val      = va_df[LABELS].astype(int).values

grid = np.linspace(0.2, 0.8, 25)
best_th, per_f1 = [], []
for i, lab in enumerate(LABELS):
    best_f, best_t = -1.0, 0.5
    p = val_probs[:, i]
    for t in grid:
        yhat = (p >= t).astype(int)
        f1 = f1_score(y_val[:, i], yhat, zero_division=0)
        if f1 > best_f:
            best_f, best_t = f1, t
    best_th.append(float(best_t)); per_f1.append(float(best_f))

val_pred_bin = (val_probs >= np.array(best_th)).astype(int)
macro_f1 = f1_score(y_val, val_pred_bin, average="macro", zero_division=0)
print("Val Macro F1:", round(macro_f1, 5))
print("Per-label F1:", dict(zip(LABELS, [round(x,4) for x in per_f1])))
print(classification_report(y_val, val_pred_bin, target_names=LABELS, zero_division=0))

# ================== Predict test & save submission ==================
test_pred   = trainer.predict(ds_test)
test_logits = test_pred.predictions
test_probs  = torch.sigmoid(torch.tensor(test_logits)).numpy()

sub = pd.DataFrame({ID_COL: test_df[ID_COL].values})
for i, lab in enumerate(LABELS):
    sub[lab] = (test_probs[:, i] >= best_th[i]).astype(int)
sub = sub[[ID_COL] + LABELS]

out_path = Path("/kaggle/working/submission.csv")
sub.to_csv(out_path, index=False)
print("Saved:", out_path)

# ================== (Optional) save artifacts & log to W&B ==================
import json as _json, pathlib, wandb
art_dir = pathlib.Path("/kaggle/working/artifacts")
(art_dir / "hf_model").mkdir(parents=True, exist_ok=True)
(art_dir / "tokenizer").mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(art_dir / "hf_model")
(getattr(trainer, "tokenizer", tokenizer)).save_pretrained(art_dir / "tokenizer")
with open(art_dir / "thresholds.json","w") as f:
    _json.dump(dict(zip(LABELS, best_th)), f, indent=2)
print("Artifacts under:", art_dir)

if wandb.run is not None:
    wandb.run.summary["val_macro_f1"] = float(macro_f1)
    for i, lab in enumerate(LABELS):
        wandb.run.summary[f"val_f1_{lab}"] = float(per_f1[i])
    wandb.run.summary["thresholds"] = {lab: float(t) for lab, t in zip(LABELS, best_th)}
    wandb.save(str(out_path))
    for p in (art_dir / "hf_model").glob("*"): wandb.save(str(p))
    for p in (art_dir / "tokenizer").glob("*"): wandb.save(str(p))
    wandb.save(str(art_dir / "thresholds.json"))
    # wandb.finish()  # optional


Val Macro F1: 0.80706
Per-label F1: {'anger': 0.7821, 'fear': 0.8371, 'joy': 0.8149, 'sadness': 0.7942, 'surprise': 0.8071}
              precision    recall  f1-score   support

       anger       0.75      0.81      0.78       150
        fear       0.81      0.86      0.84       780
         joy       0.81      0.82      0.81       334
     sadness       0.77      0.82      0.79       430
    surprise       0.78      0.84      0.81       407

   micro avg       0.79      0.84      0.81      2101
   macro avg       0.79      0.83      0.81      2101
weighted avg       0.79      0.84      0.82      2101
 samples avg       0.73      0.76      0.73      2101



Saved: /kaggle/working/submission.csv


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")


Artifacts under: /kaggle/working/artifacts


In [7]:
# K2.1 — Validate & fix submission.csv for Kaggle

import pandas as pd, numpy as np
from pathlib import Path

DATA_DIR = Path("/kaggle/input/2025-sep-dl-gen-ai-project")
TEST = DATA_DIR / "test.csv"
SAMPLE_SUB = DATA_DIR / "sample_submission.csv"
SUB_OUT = Path("/kaggle/working/submission.csv")

# 1) Load expected references
test_df = pd.read_csv(TEST)
sample  = pd.read_csv(SAMPLE_SUB)

# 2) Load your current submission (if exists)
assert SUB_OUT.exists(), "submission.csv not found at /kaggle/working/. Run K2 first."
sub = pd.read_csv(SUB_OUT)

# 3) Hard requirements
expected_cols = ["id","anger","fear","joy","sadness","surprise"]

# a) Columns exact & ordered
missing = [c for c in expected_cols if c not in sub.columns]
extra   = [c for c in sub.columns if c not in expected_cols]
if missing or extra:
    print("Fixing columns. Missing:", missing, " Extra:", extra)
    # If the shape looks right, try to realign by sample
    sub = sub[[c for c in expected_cols if c in sub.columns]]
    for c in expected_cols:
        if c not in sub.columns:
            sub[c] = 0
    sub = sub[expected_cols]

# b) Row count must match test
if len(sub) != len(test_df):
    print(f"Row count mismatch: sub={len(sub)} test={len(test_df)}. Rebuilding by join on id…")
    # If your sub has probs indexed differently, rebuild using sample ids
    # Keep existing predictions if present; otherwise fill 0
    sub = sample.copy()
    pred = pd.read_csv(SUB_OUT)
    if "id" in pred.columns:
        pred = pred.set_index("id")
        for c in expected_cols[1:]:
            if c in pred.columns:
                sub[c] = sub["id"].map(pred[c]).fillna(0)
    # else leave zeros

# c) Ensure id types & order match sample (some comps require exact order)
sub = sub.merge(sample[["id"]], on="id", how="right")  # enforce id set & order
sub = sub[expected_cols]

# d) Ensure binary 0/1 ints and no NaNs
for c in expected_cols[1:]:
    sub[c] = sub[c].clip(0,1).fillna(0).astype(int)

# e) Sanity
assert list(sub.columns) == expected_cols, "Column names/order incorrect."
assert len(sub) == len(test_df), "Row count must equal test size."
vals = set(np.unique(sub[expected_cols[1:]].values))
assert vals <= {0,1}, f"Found non-binary predictions: {vals}"

# 4) Save clean file
sub.to_csv(SUB_OUT, index=False)
print("✅ submission.csv validated & saved at:", SUB_OUT)
print(sub.head())


✅ submission.csv validated & saved at: /kaggle/working/submission.csv
   id  anger  fear  joy  sadness  surprise
0   0      1     1    0        0         0
1   1      0     0    0        0         0
2   2      1     1    0        0         1
3   3      0     1    0        0         0
4   4      0     1    0        0         1


In [8]:
# Lighter/faster baseline: TF-IDF (word 1–2 grams, 40k feats) + LinearSVC (One-vs-Rest)
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report

# Load
train_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/train.csv")
test_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/test.csv")
labels = ["anger", "fear", "joy", "sadness", "surprise"]

X = train_df["text"].fillna("")
y = train_df[labels].astype(int).values

# Split
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, shuffle=True
)

# Vectorizer
vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2, max_features=40_000, sublinear_tf=True
)
X_tr_vec = vec.fit_transform(X_tr)
X_val_vec = vec.transform(X_val)

# Model
svc = OneVsRestClassifier(LinearSVC(C=1.0))
svc.fit(X_tr_vec, y_tr)

# Validate
val_dec = svc.decision_function(X_val_vec)
val_pred = (val_dec > 0).astype(int)

macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
report = classification_report(y_val, val_pred, target_names=labels, zero_division=0)

# Train full model
X_full_vec = vec.fit_transform(X)
svc_full = OneVsRestClassifier(LinearSVC(C=1.0))
svc_full.fit(X_full_vec, y)

# Predict test
X_test_vec = vec.transform(test_df["text"].fillna(""))
test_dec = svc_full.decision_function(X_test_vec)
test_pred = (test_dec > 0).astype(int)

# Submission
sub = pd.DataFrame({"id": test_df["id"]})
for i, lab in enumerate(labels):
    sub[lab] = test_pred[:, i].astype(int)

out_path = Path("/kaggle/working/submission_svc_tfidf.csv")
sub.to_csv(out_path, index=False)

{
    "validation_macro_f1": round(float(macro_f1), 4),
    "submission_path": str(out_path),
    "submission_preview": sub.head(5).to_dict(orient="records")
}


{'validation_macro_f1': 0.748,
 'submission_path': '/kaggle/working/submission_svc_tfidf.csv',
 'submission_preview': [{'id': 0,
   'anger': 1,
   'fear': 1,
   'joy': 0,
   'sadness': 0,
   'surprise': 1},
  {'id': 1, 'anger': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0},
  {'id': 2, 'anger': 1, 'fear': 1, 'joy': 0, 'sadness': 0, 'surprise': 0},
  {'id': 3, 'anger': 0, 'fear': 1, 'joy': 0, 'sadness': 0, 'surprise': 0},
  {'id': 4, 'anger': 0, 'fear': 1, 'joy': 0, 'sadness': 0, 'surprise': 1}]}

In [9]:
# Import and Load Data
# CELL 1 — Imports & data load
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy.sparse import hstack
import pickle, json, random

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Paths

TRAIN = Path("/kaggle/input/2025-sep-dl-gen-ai-project/train.csv")
TEST = Path("/kaggle/input/2025-sep-dl-gen-ai-project/test.csv")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN)
test_df  = pd.read_csv(TEST)
labels = ["anger", "fear", "joy", "sadness", "surprise"]

print(train_df.shape, test_df.shape)
train_df.head(2)



(6827, 8) (1707, 2)


,id,text,anger,fear,joy,sadness,surprise,emotions
0,0,the dentist that did the work apparently did a...,1,0,0,1,0,['anger' 'sadness']
1,1,i'm gonna absolutely ~~suck~~ be terrible duri...,0,1,0,1,0,['fear' 'sadness']


In [10]:
# CELL A2 — Configure feature sizes & stopwords
WORD_MAX_FEATS = 80_000  # bump from 60k -> 80k
CHAR_MAX_FEATS = 60_000  # bump from 30k -> 60k
USE_EN_STOPWORDS = True  # toggle stopwords

stopwords = "english" if USE_EN_STOPWORDS else None

X = train_df["text"].fillna("")
y = train_df[labels].astype(int).values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.15, random_state=SEED, shuffle=True)

word_vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1,2), min_df=2, max_features=WORD_MAX_FEATS,
    sublinear_tf=True, stop_words=stopwords
)
char_vec = TfidfVectorizer(
    analyzer="char", ngram_range=(3,5), min_df=2, max_features=CHAR_MAX_FEATS,
    sublinear_tf=True
)

X_tr_all  = hstack([word_vec.fit_transform(X_tr),  char_vec.fit_transform(X_tr)]).tocsr()
X_val_all = hstack([word_vec.transform(X_val),      char_vec.transform(X_val)]).tocsr()
X_tr_all.shape, X_val_all.shape


((5802, 72162), (1025, 72162))

In [11]:
#  Train & tune thresholds
clf = OneVsRestClassifier(
    LogisticRegression(solver="liblinear", max_iter=200, class_weight="balanced", C=2.0)
)
clf.fit(X_tr_all, y_tr)

# Prob on val
val_proba = np.vstack([est.predict_proba(X_val_all)[:,1] for est in clf.estimators_]).T

# Tune thresholds per label
grid = np.linspace(0.1, 0.6, 26)
best_th, per_f1, cm_per = [], [], {}

for i, lab in enumerate(labels):
    best_f, best_t = -1.0, 0.5
    p = val_proba[:, i]
    for t in grid:
        pred = (p >= t).astype(int)
        f1 = f1_score(y_val[:, i], pred, zero_division=0)
        if f1 > best_f:
            best_f, best_t = f1, t
    best_th.append(float(best_t)); per_f1.append(float(best_f))

val_preds = (val_proba >= np.array(best_th)).astype(int)
macro_f1 = f1_score(y_val, val_preds, average="macro", zero_division=0)
print("Macro F1 (val):", round(macro_f1, 4))
print(classification_report(y_val, val_preds, target_names=labels, zero_division=0))

for i, lab in enumerate(labels):
    tn, fp, fn, tp = confusion_matrix(y_val[:, i], val_preds[:, i], labels=[0,1]).ravel()
    cm_per[lab] = {"tn":int(tn), "fp":int(fp), "fn":int(fn), "tp":int(tp)}

metrics = {
    "validation_macro_f1": float(macro_f1),
    "per_class_f1": dict(zip(labels, per_f1)),
    "best_thresholds": dict(zip(labels, best_th)),
    "confusion_per_label": cm_per
}
with open(WORKING/"tfidf_lr_metrics.json","w") as f:
    json.dump(metrics, f, indent=2)

metrics


Macro F1 (val): 0.769
              precision    recall  f1-score   support

       anger       0.84      0.54      0.66       121
        fear       0.83      0.90      0.86       573
         joy       0.87      0.70      0.77       243
     sadness       0.77      0.81      0.79       333
    surprise       0.77      0.75      0.76       290

   micro avg       0.81      0.79      0.80      1560
   macro avg       0.82      0.74      0.77      1560
weighted avg       0.81      0.79      0.80      1560
 samples avg       0.70      0.70      0.69      1560



{'validation_macro_f1': 0.7690385765368325,
 'per_class_f1': {'anger': 0.6565656565656566,
  'fear': 0.8616932103939647,
  'joy': 0.7734553775743708,
  'sadness': 0.7912408759124087,
  'surprise': 0.7622377622377621},
 'best_thresholds': {'anger': 0.6,
  'fear': 0.44000000000000006,
  'joy': 0.6,
  'sadness': 0.48,
  'surprise': 0.54},
 'confusion_per_label': {'anger': {'tn': 892, 'fp': 12, 'fn': 56, 'tp': 65},
  'fear': {'tn': 346, 'fp': 106, 'fn': 59, 'tp': 514},
  'joy': {'tn': 757, 'fp': 25, 'fn': 74, 'tp': 169},
  'sadness': {'tn': 611, 'fp': 81, 'fn': 62, 'tp': 271},
  'surprise': {'tn': 671, 'fp': 64, 'fn': 72, 'tp': 218}}}

In [12]:
# Train on full, export pickles, predict test
X_full = train_df["text"].fillna("")
y_full = train_df[labels].astype(int).values

word_full = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2,
                            max_features=WORD_MAX_FEATS, sublinear_tf=True,
                            stop_words=stopwords)
char_full = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=2,
                            max_features=CHAR_MAX_FEATS, sublinear_tf=True)
X_full_all = hstack([word_full.fit_transform(X_full), char_full.fit_transform(X_full)]).tocsr()

clf_full = OneVsRestClassifier(
    LogisticRegression(solver="liblinear", max_iter=250, class_weight="balanced", C=2.0)
)
clf_full.fit(X_full_all, y_full)

with open(WORKING/"tfidf_lr_metrics.json") as f: m = json.load(f)
th = np.array([m["best_thresholds"][lab] for lab in labels], float)

X_test_all = hstack([
    word_full.transform(test_df["text"].fillna("")),
    char_full.transform(test_df["text"].fillna(""))
]).tocsr()

test_proba = np.vstack([est.predict_proba(X_test_all)[:,1] for est in clf_full.estimators_]).T
test_pred  = (test_proba >= th).astype(int)

sub = pd.DataFrame({"id": test_df["id"]})
for i, lab in enumerate(labels):
    sub[lab] = test_pred[:, i].astype(int)

sub_path = WORKING/"submission_tfidf_lr_tuned.csv"
sub.to_csv(sub_path, index=False)

# export artifacts
import pickle
with open(WORKING/"tfidf_word.pkl","wb") as f: pickle.dump(word_full,f)
with open(WORKING/"tfidf_char.pkl","wb") as f: pickle.dump(char_full,f)
with open(WORKING/"tfidf_lr_ovr.pkl","wb") as f: pickle.dump(clf_full,f)
with open(WORKING/"tfidf_thresholds.json","w") as f: json.dump(m["best_thresholds"], f, indent=2)

sub.head()


,id,anger,fear,joy,sadness,surprise
0,0,1,0,1,0,1
1,1,0,0,0,0,0
2,2,1,1,0,0,0
3,3,0,1,0,0,0
4,4,0,1,0,0,1


B) RoBERTa-base — Multi-label BCE + Threshold Tuning

If internet is off: attach a dataset containing roberta-base folder (from HF) and set MODEL_PATH = Path("/kaggle/input/your-roberta-dataset/roberta-base").
Otherwise MODEL_PATH = "roberta-base" will download.

In [13]:
# Silence TF/JAX/XLA noise BEFORE importing tensorflow/jax/transformers
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # TensorFlow C++ logs -> ERROR only
os.environ["JAX_PLATFORM_NAME"] = "gpu"      # be explicit; avoids extra probing
os.environ["NVIDIA_TF32_OVERRIDE"] = "0"     # optional; keeps default behavior
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # reduce XLA prealloc noise

# Optional: cut down Python-side logs
try:
    from absl import logging as absl_logging
    absl_logging.set_verbosity(absl_logging.ERROR)
except Exception:
    pass


In [14]:
# CELL B0 — Imports and paths
import os, json, random
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset
import torch
from torch.nn import BCEWithLogitsLoss
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Model source: local path or HF id
MODEL_PATH = "roberta-base"  # or Path("/kaggle/input/roberta-base")

MAX_LEN = 192
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

labels = ["anger","fear","joy","sadness","surprise"]
id2label = {i:l for i,l in enumerate(labels)}
label2id = {l:i for i,l in enumerate(labels)}

train_df = pd.read_csv(TRAIN)
test_df  = pd.read_csv(TEST)


In [15]:
# CELL B1 — Tokenize & build datasets
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)

def preprocess(df, is_train=True):
    enc = tokenizer(
        df["text"].tolist(), truncation=True, padding="max_length", max_length=MAX_LEN
    )
    if is_train:
        y = df[labels].astype(float).values
        enc["labels"] = y
    return enc

# Split
from sklearn.model_selection import train_test_split
tr_df, val_df = train_test_split(train_df, test_size=0.15, random_state=SEED, shuffle=True)

ds_train = Dataset.from_dict(preprocess(tr_df, is_train=True))
ds_val   = Dataset.from_dict(preprocess(val_df, is_train=True))
ds_test  = Dataset.from_dict(tokenizer(test_df["text"].tolist(), truncation=True, padding="max_length", max_length=MAX_LEN))

ds_train, ds_val


(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 5802
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 1025
 }))

In [16]:
# CELL B2 — Model setup (multi-label)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=len(labels),
    problem_type="multi_label_classification",
    id2label=id2label, label2id=label2id
)

# Optional: class imbalance handling (pos_weight)
# Compute per-class pos_weight = (N - pos) / pos
y_all = train_df[labels].astype(int).values
pos = y_all.sum(axis=0)
neg = y_all.shape[0] - pos
pos_weight = torch.tensor((neg / np.clip(pos, 1, None))).float()

# Custom loss to inject pos_weight (Trainer hook)
def custom_loss(model, inputs, return_outputs=False):
    labels_t = inputs.pop("labels")
    outputs = model(**inputs)
    logits = outputs.logits
    loss = BCEWithLogitsLoss(pos_weight=pos_weight.to(logits.device))(logits, labels_t)
    return (loss, outputs) if return_outputs else loss


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ELECTRA-base — Multi-label BCE + Threshold Tuning

If offline, attach google/electra-base-discriminator as dataset and set MODEL_PATH to that local directory.

In [17]:
# CELL C0 — Imports and paths
import os, json, random
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset
import torch
from torch.nn import BCEWithLogitsLoss
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_PATH = "google/electra-base-discriminator"  # or local Path to dataset
MAX_LEN = 192
BATCH_SIZE = 16
EPOCHS = 3
LR = 3e-5

labels = ["anger","fear","joy","sadness","surprise"]
id2label = {i:l for i,l in enumerate(labels)}
label2id = {l:i for i,l in enumerate(labels)}

train_df = pd.read_csv(TRAIN)
test_df  = pd.read_csv(TEST)


In [18]:
# CELL C1 — Tokenize & build datasets
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)

def preprocess(df, is_train=True):
    enc = tokenizer(
        df["text"].tolist(), truncation=True, padding="max_length", max_length=MAX_LEN
    )
    if is_train:
        enc["labels"] = df[labels].astype(float).values
    return enc

from sklearn.model_selection import train_test_split
tr_df, val_df = train_test_split(train_df, test_size=0.15, random_state=SEED, shuffle=True)

ds_train = Dataset.from_dict(preprocess(tr_df, is_train=True))
ds_val   = Dataset.from_dict(preprocess(val_df, is_train=True))
ds_test  = Dataset.from_dict(tokenizer(test_df["text"].tolist(), truncation=True, padding="max_length", max_length=MAX_LEN))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [19]:
# CELL C2 — Model (multi-label) and custom loss with pos_weight
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=len(labels),
    problem_type="multi_label_classification",
    id2label=id2label, label2id=label2id
)

y_all = train_df[labels].astype(int).values
pos = y_all.sum(axis=0); neg = y_all.shape[0] - pos
pos_weight = torch.tensor((neg / np.clip(pos, 1, None))).float()

def custom_loss(model, inputs, return_outputs=False):
    y = inputs.pop("labels")
    out = model(**inputs)
    logits = out.logits
    loss = BCEWithLogitsLoss(pos_weight=pos_weight.to(logits.device))(logits, y)
    return (loss, out) if return_outputs else loss


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at google/electra-base-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]